# Can we predict 12-month loan default?

**Question:** using only information available at application time, can we predict `defaulted` (1 = defaulted within 12 months of funding)?

Plan: inspect the data → check for quality issues → baseline → logistic regression and gradient boosting with cross-validation → one final evaluation on a held-out test set → interpret and state the practical value.

Metrics: the target is imbalanced, so accuracy is misleading. Primary metric is **ROC AUC** (ranking quality) with **PR AUC** (compared with the base rate) as a secondary one, plus a lift/decile analysis.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve, brier_score_loss
from sklearn.inspection import permutation_importance
from sklearn.calibration import calibration_curve
pd.set_option('display.width',140)
SEED=42

## 1. Data and dictionary

In [ ]:
df=pd.read_csv('data.csv'); cols=pd.read_csv('columns.csv')
display(cols)
print(df.shape); display(df.head()); display(df.describe().T)
print('missing values:',int(df.isna().sum().sum()),'| duplicate rows:',int(df.duplicated().sum()),'| duplicate ids:',int(df.application_id.duplicated().sum()))

In [ ]:
rate=df.defaulted.mean(); print(f'Default rate: {rate:.2%}  ({df.defaulted.sum()} of {len(df)})')

Data is clean (no missing values or duplicates). About 1 in 10 loans default, so a model that always predicts "no default" is ~89% accurate, which is why we use AUC / PR AUC.

`application_id` is an identifier, so I drop it from the features (its correlation with the target is checked below as a sanity test). The dataset has no application date, so I can't do a time-based split; I use stratified random splits and note this as a limitation.

## 2. Exploratory look: default rate by feature

In [ ]:
num=['income_k','debt_ratio','credit_lines','age','employment_years','loan_amount_k']
fig,ax=plt.subplots(2,3,figsize=(14,7))
for a,c in zip(ax.ravel(),num):
    q=pd.qcut(df[c],10,duplicates='drop')
    g=df.groupby(q,observed=True).defaulted.mean()
    a.plot(range(len(g)),g.values,marker='o'); a.axhline(rate,ls='--',c='gray')
    a.set_title(c); a.set_xlabel('decile of feature (low→high)'); a.set_ylabel('default rate')
plt.tight_layout(); plt.show()
print(df.groupby('region').defaulted.agg(['mean','size']))
print('\nSpearman corr. with target:'); print(df[num+['application_id']].corrwith(df.defaulted,method='spearman').round(3))

## 3. Split
20% held-out test set, touched only once at the end. All model selection uses 5-fold stratified CV on the training portion.

In [ ]:
X=df.drop(columns=['defaulted','application_id']); y=df.defaulted
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=0.2,stratify=y,random_state=SEED)
print(Xtr.shape,Xte.shape,ytr.mean().round(4),yte.mean().round(4))

## 4. Models (cross-validated on train)

A flexible model alongside the linear one shows whether non-linearities or interactions matter: if the two perform about the same, the signal is mostly additive.

In [ ]:
cat=['region']
lin_pre=ColumnTransformer([('n',StandardScaler(),num),('c',OneHotEncoder(drop='first'),cat)])
gb_pre=ColumnTransformer([('n','passthrough',num),('c',OneHotEncoder(),cat)])
models={
 'baseline (prior)':Pipeline([('m',DummyClassifier(strategy='prior'))]),
 'logistic regression':Pipeline([('p',lin_pre),('m',LogisticRegression(max_iter=1000))]),
 'gradient boosting':Pipeline([('p',gb_pre),('m',HistGradientBoostingClassifier(learning_rate=0.05,max_iter=200,max_depth=3,min_samples_leaf=40,l2_regularization=1.0,random_state=SEED))]),
}
cv=StratifiedKFold(5,shuffle=True,random_state=SEED)
rows={}
for n,m in models.items():
    r=cross_validate(m,Xtr,ytr,cv=cv,scoring={'roc_auc':'roc_auc','pr_auc':'average_precision','brier':'neg_brier_score'})
    rows[n]={'ROC AUC':r['test_roc_auc'].mean(),'AUC std':r['test_roc_auc'].std(),'PR AUC':r['test_pr_auc'].mean(),'Brier':-r['test_brier'].mean()}
cvres=pd.DataFrame(rows).T; display(cvres.round(4))

## 5. Final evaluation on the held-out test set

In [ ]:
fitted={n:m.fit(Xtr,ytr) for n,m in models.items()}
P={n:m.predict_proba(Xte)[:,1] for n,m in fitted.items()}
test=pd.DataFrame({n:{'ROC AUC':roc_auc_score(yte,p),'PR AUC':average_precision_score(yte,p),'Brier':brier_score_loss(yte,p)} for n,p in P.items()}).T
display(test.round(4)); print('Test base rate (= PR AUC of a random ranker): %.4f'%yte.mean())

rng=np.random.default_rng(SEED); p=P['logistic regression']; yv=yte.values
bs=[roc_auc_score(yv[i],p[i]) for i in (rng.integers(0,len(yv),len(yv)) for _ in range(1000))]
print('Logistic regression test AUC 95%% bootstrap CI: [%.3f, %.3f]'%tuple(np.percentile(bs,[2.5,97.5])))

In [ ]:
fig,ax=plt.subplots(1,3,figsize=(16,4.5))
for n,p in P.items():
    f,t,_=roc_curve(yte,p); ax[0].plot(f,t,label=f'{n} ({roc_auc_score(yte,p):.3f})')
    pr,rc,_=precision_recall_curve(yte,p); ax[1].plot(rc,pr,label=n)
ax[0].plot([0,1],[0,1],'k:'); ax[0].set(title='ROC',xlabel='FPR',ylabel='TPR'); ax[0].legend()
ax[1].axhline(yte.mean(),c='k',ls=':'); ax[1].set(title='Precision-recall',xlabel='recall',ylabel='precision'); ax[1].legend()
for n in ['logistic regression','gradient boosting']:
    fo,mp=calibration_curve(yte,P[n],n_bins=8,strategy='quantile'); ax[2].plot(mp,fo,marker='o',label=n)
ax[2].plot([0,.4],[0,.4],'k:'); ax[2].set(title='Calibration',xlabel='predicted',ylabel='observed'); ax[2].legend()
plt.tight_layout(); plt.show()

### Business view: what do the scores buy us?

In [ ]:
d=pd.DataFrame({'y':yte.values,'p':P['logistic regression']})
d['decile']=pd.qcut(d.p.rank(method='first'),10,labels=range(1,11))  # 10 = riskiest
t=d.groupby('decile',observed=True).agg(n=('y','size'),default_rate=('y','mean'),mean_pred=('p','mean'))
t['lift']=t.default_rate/d.y.mean(); display(t.round(3).iloc[::-1])
top=d[d.decile>=9]; print('Riskiest 20%% of applicants hold %.0f%% of all defaults (random would be 20%%).'%(100*top.y.sum()/d.y.sum()))
print('Safest 50%% hold %.0f%% of defaults.'%(100*d[d.decile<=5].y.sum()/d.y.sum()))

## 6. What drives the prediction?

In [ ]:
lr=fitted['logistic regression']
names=lr.named_steps['p'].get_feature_names_out()
coef=pd.Series(lr.named_steps['m'].coef_[0],index=names).sort_values()
print('Logistic coefficients (numeric features standardised, so comparable):'); display(coef.round(3))
pi=permutation_importance(fitted['gradient boosting'],Xte,yte,scoring='roc_auc',n_repeats=20,random_state=SEED)
imp=pd.Series(pi.importances_mean,index=Xte.columns).sort_values()
imp.plot.barh(xerr=pd.Series(pi.importances_std,index=Xte.columns)[imp.index],title='Permutation importance (drop in test AUC), gradient boosting'); plt.show()

## 7. Conclusions

*(Figures below are from running this notebook with `SEED=42`; re-check them if you change the seed or data.)*

**Can we predict default? Yes, moderately.**
- Held-out ROC AUC is about **0.775** (bootstrap 95% CI roughly 0.74–0.81) versus 0.50 for the baseline. PR AUC is about 0.35 versus a 0.104 base rate, a ~3x improvement.
- The model is far better at *ranking* risk than at pinpointing individual defaulters. The riskiest decile defaults at ~36% against ~10% overall, and the safest two deciles at ~1%. The riskiest 20% of applicants hold roughly half of all defaults. It is useful for risk-based pricing, screening or prioritising manual review, but not for confident individual yes/no calls (precision is limited).
- Logistic regression is as good as or better than gradient boosting (CV AUC 0.776 vs 0.761), so the signal is mostly additive and monotone; extra complexity does not help. Probabilities are reasonably calibrated.

**Drivers:** higher `debt_ratio`, lower `income_k`, shorter `employment_years` and larger `loan_amount_k` raise risk. `region`, `age` and `credit_lines` carry essentially no signal. `application_id` is uncorrelated with the target (as expected for an identifier).

**Caveats**
- No application date is available, so the split is random, not out-of-time. Performance on future cohorts could be lower if the population or economy shifts.
- Only funded loans have outcomes, so the data presumably reflects past approval decisions (selection bias); the model may not hold for applicants who would have been rejected.
- 837 defaults is a small sample; test AUC has a ±0.03 uncertainty. A fixed score cutoff should be chosen from the business cost of a missed default versus a rejected good loan, which we don't know.
- If `debt_ratio`/`income_k` interact in the real world (e.g. loan size relative to income), engineered ratios may add a little; not explored here.
